# Laboratoire 1 : Sphère de Bloch et premiers circuits quantiques

**Quantum Machine Learning — Master/PhD**

Semaine 2 — Fondements du calcul quantique

## Objectifs

- Maîtriser la représentation des qubits sur la sphère de Bloch
- Implémenter et visualiser les portes quantiques fondamentales
- Créer des états intriqués (États de Bell) et mesurer
- Se familiariser avec PennyLane et Qiskit pour la construction de circuits

## Bibliothèques requises

- `pennylane` (≥ 0.45)
- `qiskit` (≥ 2.0)
- `qiskit-aer`
- `matplotlib`
- `numpy`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp

# Imports Qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit.quantum_info import Statevector

print('Bibliothèques chargées avec succès.')
print(f'PennyLane version : {qml.__version__}')

---
## Partie A : Sphère de Bloch

La sphère de Bloch représente l'état d'un qubit $|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$.

Nous allons visualiser les états de base : $|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$.

In [ ]:
# Visualisation des états fondamentaux avec Qiskit

def etat_sur_sphere(vecteur, titre):
    """Affiche un état sur la sphère de Bloch."""
    qc = QuantumCircuit(1)
    qc.initialize(vecteur, 0)
    state = Statevector(qc)
    fig = plot_bloch_multivector(state, title=titre)
    return fig

# États de base
ket0 = [1, 0]  # |0>
ket1 = [0, 1]  # |1>
ket_plus = [1/np.sqrt(2), 1/np.sqrt(2)]   # |+> = (|0> + |1>)/sqrt(2)
ket_minus = [1/np.sqrt(2), -1/np.sqrt(2)] # |-> = (|0> - |1>)/sqrt(2)

etat_sur_sphere(ket0, r'$|0\rangle$')
plt.show()
etat_sur_sphere(ket1, r'$|1\rangle$')
plt.show()
etat_sur_sphere(ket_plus, r'$|+\rangle$')
plt.show()
etat_sur_sphere(ket_minus, r'$|-\rangle$')
plt.show()

In [ ]:
# Visualisation avec PennyLane (BlochSphere intégré)

dev = qml.device('default.qubit', wires=1)

@qml.qnode(dev)
def circuit_bloch():
    # L'état |0> est l'état par défaut
    return qml.state()

# Afficher la sphère de Bloch pour |0>
print("État |0> sur la sphère de Bloch (PennyLane) :")
qml.draw_mpl(circuit_bloch)()
plt.show()

# Créons |+> avec une porte Hadamard
@qml.qnode(dev)
def circuit_plus():
    qml.Hadamard(wires=0)
    return qml.state()

print("État |+> = H|0> :")
fig, ax = qml.draw_mpl(circuit_plus)()
plt.show()
print(f"Vecteur d'état : {circuit_plus()}")

---
## Partie B : Portes quantiques et rotations

Les portes $R_X(\theta)$, $R_Y(\theta)$, $R_Z(\theta)$ effectuent des rotations autour des axes de la sphère de Bloch.
La porte Hadamard crée des superpositions. La porte CNOT intrique deux qubits.

In [ ]:
# Rotations RX, RY, RZ avec PennyLane

dev = qml.device('default.qubit', wires=1)

@qml.qnode(dev)
def rotation_rx(theta):
    qml.RX(theta, wires=0)
    return qml.state()

@qml.qnode(dev)
def rotation_ry(theta):
    qml.RY(theta, wires=0)
    return qml.state()

@qml.qnode(dev)
def rotation_rz(theta):
    qml.RZ(theta, wires=0)
    return qml.state()

# Testons avec un angle de pi/2
angles = [0, np.pi/4, np.pi/2, np.pi, 2*np.pi]
print("Rotation RX(θ) :")
for th in angles:
    psi = rotation_rx(th)
    print(f"  RX({th:.2f}) -> |ψ⟩ = [{psi[0]:.3f}, {psi[1]:.3f}]")

print("\nRotation RY(θ) :")
for th in angles:
    psi = rotation_ry(th)
    print(f"  RY({th:.2f}) -> |ψ⟩ = [{psi[0]:.3f}, {psi[1]:.3f}]")

In [ ]:
# Visualisation des rotations sur la sphère de Bloch avec Qiskit

def visualiser_rotation(portes, titre):
    qc = QuantumCircuit(1)
    for porte, params in portes:
        if porte == 'rx':
            qc.rx(params, 0)
        elif porte == 'ry':
            qc.ry(params, 0)
        elif porte == 'rz':
            qc.rz(params, 0)
        elif porte == 'h':
            qc.h(0)
    state = Statevector(qc)
    fig = plot_bloch_multivector(state, title=titre)
    return fig

# Exemple : |0> -> RY(pi/2) -> RX(pi) -> RZ(pi/4)
fig = visualiser_rotation(
    [('ry', np.pi/2), ('rx', np.pi), ('rz', np.pi/4)],
    r'$R_Y(\pi/2) \, R_X(\pi) \, R_Z(\pi/4) \, |0\rangle$'
)
plt.show()

# Comparaison : Hadamard vs RY(pi/2) + RX(pi)
fig2 = visualiser_rotation([('h', 0)], r'$H|0\rangle = |+\rangle$')
plt.show()
fig3 = visualiser_rotation([('ry', np.pi/2)], r'$R_Y(\pi/2)|0\rangle$')
plt.show()

In [ ]:
# Représentation matricielle des portes

# Matrices de Pauli
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)

# Porte Hadamard
H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)

# Porte CNOT (2 qubits)
CNOT = np.array([[1, 0, 0, 0],
                 [0, 1, 0, 0],
                 [0, 0, 0, 1],
                 [0, 0, 1, 0]], dtype=complex)

print("Matrice de Pauli X (NOT) :")
print(sx)
print("\nMatrice de Hadamard :")
print(H)
print("\nMatrice CNOT :")
print(CNOT)

# Vérifions : H|0> = |+>
ket0_vec = np.array([1, 0], dtype=complex)
ket_plus_calc = H @ ket0_vec
print(f"\nH|0> = {ket_plus_calc}")

---
## Partie C : États de Bell et intrication

Les états de Bell sont des états maximally intriqués à 2 qubits :

- $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$
- $|\Phi^-\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$
- $|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)$
- $|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$

In [ ]:
# Création des états de Bell avec PennyLane

dev2 = qml.device('default.qubit', wires=2, shots=1000)

@qml.qnode(dev2)
def bell_phi_plus():
    """|Φ^+> = (|00> + |11>)/sqrt(2)"""
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.state()

@qml.qnode(dev2)
def bell_phi_minus():
    """|Φ^-> = (|00> - |11>)/sqrt(2)"""
    qml.Hadamard(wires=0)
    qml.PauliZ(wires=1)  # phase
    qml.CNOT(wires=[0, 1])
    return qml.state()

@qml.qnode(dev2)
def bell_psi_plus():
    """|Ψ^+> = (|01> + |10>)/sqrt(2)"""
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    qml.PauliX(wires=0)
    return qml.state()

@qml.qnode(dev2)
def bell_psi_minus():
    """|Ψ^-> = (|01> - |10>)/sqrt(2)"""
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    qml.PauliX(wires=0)
    qml.PauliZ(wires=0)
    return qml.state()

# Affichage
etats = [
    (r'$|\Phi^+\rangle$', bell_phi_plus()),
    (r'$|\Phi^-\rangle$', bell_phi_minus()),
    (r'$|\Psi^+\rangle$', bell_psi_plus()),
    (r'$|\Psi^-\rangle$', bell_psi_minus()),
]
for nom, psi in etats:
    print(f"{nom} = {np.round(psi, 3)}")

In [ ]:
# Mesure et distribution de probabilités

@qml.qnode(dev2)
def mesurer_bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.counts()

counts = mesurer_bell()
print("Distribution des mesures pour |Φ^+> :")
total = sum(counts.values())
for etat, count in sorted(counts.items()):
    prob = count / total
    print(f"  |{etat}⟩ : {count}/{total} = {prob:.3f}")

# Visualisation avec Qiskit
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure_all()

sim = AerSimulator()
result = sim.run(qc_bell, shots=1000).result()
counts_qiskit = result.get_counts()
print("\nAvec Qiskit :")
for etat, count in sorted(counts_qiskit.items()):
    print(f"  |{etat}⟩ : {count}/1000 = {count/1000:.3f}")

plot_histogram(counts_qiskit, title='Distribution des mesures |Φ^+>')
plt.show()

---
## Partie D : Exercices

### Exercice 1 : Créer l'état $|+\rangle$ avec des rotations

Utilisez uniquement des portes $R_X$, $R_Y$, $R_Z$ pour créer l'état $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ à partir de $|0\rangle$.

*Indice* : $R_Y(\pi/2)|0\rangle = |+\rangle$.

In [ ]:
# Votre code pour l'exercice 1
dev_ex1 = qml.device('default.qubit', wires=1)

@qml.qnode(dev_ex1)
def creer_etat_plus():
    # TODO : utiliser RY(pi/2) pour créer |+>
    pass  # Remplacez ceci

# Test
# print(creer_etat_plus())

### Exercice 2 : Protocole de téléportation quantique simple

Implémentez le circuit de téléportation quantique :
1. Préparation d'un état $|\psi\rangle = R_Y(\theta)|0\rangle$ sur le qubit 0
2. Intrication entre les qubits 1 et 2 (Alice et Bob)
3. Mesure de Bell sur les qubits 0 et 1
4. Correction conditionnelle de Bob (qubit 2)

In [ ]:
# Votre code pour l'exercice 2 — Téléportation quantique

def teleportation(theta):
    """
    Téléporte l'état RY(theta)|0> du qubit 0 au qubit 2.
    """
    dev_tele = qml.device('default.qubit', wires=3)

    @qml.qnode(dev_tele)
    def circuit():
        # 1. Préparer l'état à téléporter sur q0
        qml.RY(theta, wires=0)

        # 2. Créer une paire intriquée entre q1 et q2
        qml.Hadamard(wires=1)
        qml.CNOT(wires=[1, 2])

        # 3. Mesure de Bell sur q0 et q1
        qml.CNOT(wires=[0, 1])
        qml.Hadamard(wires=0)

        # 4. Correction conditionnelle
        qml.CNOT(wires=[1, 2])
        qml.CZ(wires=[0, 2])

        return qml.state()

    return circuit()

# Test de la téléportation
theta_test = 0.8
etat_final = teleportation(theta_test)
print(f"État final (téléportation de RY({theta_test})|0⟩) :")
print(np.round(etat_final.reshape(8), 3))

# L'état téléporté est dans le sous-espace du qubit 2
print("\nL'état du qubit 2 (Bob) devrait être RY({})|0⟩.".format(theta_test))

### Exercice 3 : Visualisation d'un état aléatoire

Créez un état aléatoire sur la sphère de Bloch en appliquant $R_X(\alpha)R_Y(\beta)R_Z(\gamma)$ avec des angles aléatoires.

In [ ]:
# Exercice 3 : état aléatoire
import random

alpha = random.uniform(0, 2*np.pi)
beta = random.uniform(0, 2*np.pi)
gamma = random.uniform(0, 2*np.pi)

qc_rand = QuantumCircuit(1)
qc_rand.rx(alpha, 0)
qc_rand.ry(beta, 0)
qc_rand.rz(gamma, 0)

state = Statevector(qc_rand)
fig = plot_bloch_multivector(state, title=f'État aléatoire : RX({alpha:.2f}) RY({beta:.2f}) RZ({gamma:.2f})')
plt.show()
print(f"Vecteur d'état : {np.round(state.data, 3)}")

---
## Pour aller plus loin

- Expérimentez avec des circuits à 3 qubits et l'état GHZ : $|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$
- Comparez les distributions de probabilités pour différents nombres de shots (100, 1000, 10000)
- Implémentez les portes Toffoli (CCNOT) et SWAP
- Consultez : [SP21] Ch. 3 (Schuld & Petruccione), [NC00] Ch. 1-2 (Nielsen & Chuang)